# Notebook 05 — Expert Model Selection

This notebook is step 5 of the 9-notebook pipeline of the Mixture-of-Experts (MoE) forecaster for the hourly electricity demand of Colombia's National Interconnected System (SIN):

01 data acquisition (XM S.A. SINERGOX API) → 02 preprocessing and feature engineering → 03 clustering of daily and weekly demand profiles → 04 cluster analysis and cluster classifiers → **05 expert selection** → 06 gating design → 07 hyperparameter optimization (HPO) → 08 production assembly → 09 benchmark.

In the MoE architecture, each *expert* is a regression model specialised in one combination of day type and week type, i.e. one pair (day cluster, week cluster) obtained in Notebook 03. This notebook decides which regression model is used as the expert for each pair.

---

## Objective

Select the regression model that acts as the expert for each cluster combination (`expert_id`, e.g. `day0_week1`). Three model families are evaluated — linear models, tree-based machine-learning (ML) ensembles and deep-learning (DL) networks — with five candidate models per family (up to 15 candidates per expert). All candidates are scored with walk-forward (expanding-window) cross-validation, subject to bias–variance constraints that limit model complexity according to the number of observations available for each expert.

Two selection strategies are compared:

- **Homogeneous**: all experts use models from the same family (the best model of that family is chosen for each expert).
- **Heterogeneous**: each expert uses the best model across all families allowed for it.

The comparison is made with a global MASE weighted by expert (cluster) size.

## Inputs

| File | Produced by | Content |
|:---|:---|:---|
| `DATA_DIR/processed/features.parquet` | Notebook 02 | Hourly demand and engineered features (calendar features and demand lags) |
| `DATA_DIR/clustering/cluster_assignments.parquet` | Notebook 03 | Hourly `datetime` mapped to `cluster_day`, `cluster_week` and `expert_id` |

The cluster classifiers trained in Notebook 04 are **not** used here: experts are trained and evaluated on the true (clustering-derived) cluster assignments. The classifiers are used for routing in Notebook 06.

## Outputs

| File | Content | Used by |
|:---|:---|:---|
| `RESULTS_DIR/05_expert_selection.csv` | Cross-validation metrics of every (expert, model) evaluation, homogeneous and heterogeneous | Documentation |
| `RESULTS_DIR/05_expert_selection_best.csv` | Selected (best) model per expert with its metrics | Notebooks 07 and 08 |
| `RESULTS_DIR/05_mase_global_comparison.csv` | Global MASE of each selection strategy | Documentation |
| `MODELS_DIR/expert_models.pkl` | Dictionary `expert_id → {model, model_name, family, n_obs}` with the fitted experts | Backward compatibility |
| `MODELS_DIR/experts/expert_<expert_id>.pkl` | One fitted expert per file (e.g. `expert_day0_week1.pkl`) | Notebook 06 |
| `RESULTS_DIR/05_*.png` | Figures of Section 9 | Documentation |

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

## 1. Imports and configuration

Libraries, global random seed (`RANDOM_SEED = 42`, applied to NumPy and PyTorch), plotting style and the PyTorch device (GPU if available, otherwise CPU).

In [ ]:
import os
import warnings
import pickle
from pathlib import Path
from copy import deepcopy
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor,
)
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS = -1

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Plot style ─────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.figsize": (12, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
PALETTE = sns.color_palette("Set2", 8)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
print(f"Random seed:    {RANDOM_SEED}")

---

## 2. Data loading

### 2.1 Features and cluster assignments

The hourly feature matrix (Notebook 02) and the hourly cluster assignments (Notebook 03) are loaded.

In [ ]:
# ── Load features and cluster assignments ───────────────────────────────────────────
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
clusters_path = os.path.join(DATA_DIR, "clustering", "cluster_assignments.parquet")

df_feat = pd.read_parquet(features_path)
df_clust = pd.read_parquet(clusters_path)

print(f"Feature matrix shape:      {df_feat.shape}")
print(f"Cluster assignments shape: {df_clust.shape}")
print(f"\nFeature columns ({len(df_feat.columns)}):")
print(df_feat.columns.tolist()[:20], "..." if len(df_feat.columns) > 20 else "")
print(f"\nCluster assignment columns: {df_clust.columns.tolist()}")
display(df_clust.head())

In [ ]:
df_feat.head()

### 2.2 Exclusion of 2020 and 2021

Observations from the years 2020 and 2021 are removed from the feature matrix before any model is fitted, so expert selection is based on 2022 onwards only.

In [ ]:
# Exclude the years 2020 and 2021 (see Section 2.2)
df_feat = df_feat[~df_feat.index.year.isin([2020, 2021])]

### 2.3 Attaching the expert identifier

Each hourly observation is joined, by `datetime`, with its `expert_id` (the `(cluster_day, cluster_week)` pair from Notebook 03). If the assignments file did not contain an `expert_id` column, it would be built from the day/week cluster columns. Rows that cannot be matched to an expert are dropped.

In [ ]:
df_clust = pd.read_parquet(clusters_path)

# Make 'datetime' a column of df_feat with datetime dtype
df_feat = df_feat.reset_index() # Turn the datetime index into a column
df_feat["datetime"] = pd.to_datetime(df_feat["datetime"]) # Ensure datetime dtype

# Merge key (the 'datetime' column of df_feat)
feat_key = "datetime"

# The following block that manipulated df_clust['datetime'] is disabled because
# df_clust already contains a correctly typed 'datetime' column.
# if len(df_clust) <= len(df_feat):
#     df_clust['datetime'] = df_feat['datetime'].iloc[:len(df_clust)].values
# else:
#     df_clust = df_clust.iloc[:len(df_feat)].copy()
#     df_clust['datetime'] = df_feat['datetime'].values

# Build 'expert_id' if the assignments file does not already contain it.
if "expert_id" not in df_clust.columns:
    td_col = [c for c in df_clust.columns if "type_day" in c.lower() or "cluster_day" in c.lower()]
    tw_col = [c for c in df_clust.columns if "type_week" in c.lower() or "cluster_week" in c.lower()]

    if td_col and tw_col:
        df_clust["expert_id"] = df_clust[td_col[0]].astype(str) + "_" + df_clust[tw_col[0]].astype(str)
    elif td_col:
        df_clust["expert_id"] = df_clust[td_col[0]].astype(str)
    else:
        # Fallback when no day/week cluster columns are found
        print("WARNING: 'cluster_day' / 'cluster_week' not found; expert_id cannot be built. A single default expert will be used.")
        df_clust["expert_id"] = "default_expert"


# Left merge: keep every row of df_feat and attach 'expert_id' from df_clust
df = df_feat.merge(
    df_clust[[feat_key, "expert_id"]],
    on=feat_key,
    how="left", # left merge preserves all rows of df_feat
)

if df['expert_id'].isnull().any():
    print("WARNING: some rows of df_feat could not be matched to an expert_id. These rows will be dropped.")
    original_shape = df.shape[0]
    df.dropna(subset=['expert_id'], inplace=True)
    if df.shape[0] < original_shape:
        print(f"Dropped {original_shape - df.shape[0]} rows with missing expert_id after the merge.")

df = df.sort_values(feat_key).reset_index(drop=True)

print(f"Shape after merge: {df.shape}")
if not df.empty:
    print(f"Expert IDs:   {sorted(df['expert_id'].unique())}")
    print(f"\nObservations per expert:")
    print(df["expert_id"].value_counts().sort_index())
else:
    print("WARNING: the merged DataFrame 'df' is empty.")

In [ ]:
df_clust.head()

### 2.4 Target and feature set

The feature set used by all experts consists of the cyclical calendar encodings (hour, day of week, month), the holiday indicators (`is_holiday`, `is_bridge`, i.e. bridge holiday, and the distances in days to the next/previous holiday) and all demand lags (`demand_lag_*`). Rolling-window statistics and other columns of the feature matrix are not used. The target is the hourly demand column (`demand`).

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df_feat.columns)

# Demand lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
# ── Identify target column and feature columns ────────────────────────────────────────────────
TARGET = None
for candidate in ["demand", "demanda", "target", "y", "load", "demand_kwh"]:
    if candidate in df.columns:
        TARGET = candidate
        break
if TARGET is None:
    # Heuristic fallback: first numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    TARGET = numeric_cols[0]
    print(f"WARNING: target column auto-detected as '{TARGET}'. Please verify it is correct.")

EXCLUDE_COLS = {feat_key, TARGET, "expert_id"}
#FEATURE_COLS = [
#    c for c in df.columns
#    if c not in EXCLUDE_COLS and df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
#]

FEATURE_COLS = calendar_lags

print(f"Target column:   {TARGET}")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Features:        {FEATURE_COLS}")

---

## 3. Evaluation metrics

All metrics are implemented in this notebook so that it is self-contained:

- **RMSE** and **MAE** in the units of the demand series.
- **MAPE**, returned here as a *fraction* (multiply by 100 for percent); observations with zero demand are excluded.
- **MASE** (Mean Absolute Scaled Error): the MAE of the forecast divided by the MAE of a seasonal-naive reference with a lag of 168 positions (one week of hourly data), computed on the same evaluated segment:

$$\text{MASE} = \frac{\frac{1}{n}\sum_t |y_t - \hat y_t|}{\frac{1}{n-168}\sum_{t>168} |y_t - y_{t-168}|}$$

  Because each expert's sample contains only the hours that belong to its cluster combination, consecutive rows are not always consecutive hours; the 168-position lag is therefore a within-sample seasonal reference rather than an exact calendar-week lag.

- **Global MASE**, used to compare selection strategies, is the average of the per-expert MASE values weighted by the number of observations of each expert:

$$\text{MASE}_{\text{global}} = \sum_k \frac{n_k}{N}\,\text{MASE}_k$$

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Metrics — fully self-contained (no external metric imports)
# ══════════════════════════════════════════════════════════════════════════════

def calc_rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def calc_mae(y_true, y_pred):
    """Mean Absolute Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def calc_mape(y_true, y_pred):
    """Mean Absolute Percentage Error (returned as a fraction, not in percent)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return float("inf")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])))


def calc_mase(y_true, y_pred, seasonal_period=168):
    """Mean Absolute Scaled Error (seasonal-naive reference with lag 168 = one week of hourly data)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    naive_errors = np.abs(y_true[seasonal_period:] - y_true[:-seasonal_period])
    naive_mae = np.mean(naive_errors) if len(naive_errors) > 0 else 0.0
    if naive_mae == 0:
        return float("inf")
    forecast_mae = np.mean(np.abs(y_true - y_pred))
    return float(forecast_mae / naive_mae)


def calc_all_metrics(y_true, y_pred):
    """Return a dictionary with RMSE, MAE, MAPE and MASE."""
    return {
        "rmse": calc_rmse(y_true, y_pred),
        "mae": calc_mae(y_true, y_pred),
        "mape": calc_mape(y_true, y_pred),
        "mase": calc_mase(y_true, y_pred),
    }


def weighted_mase_global(mase_per_expert, n_per_expert):
    """MASE_global = sum_k (n_k / N) * MASE_k, i.e. weighted by cluster (expert) size."""
    total_n = sum(n_per_expert.values())
    if total_n == 0:
        return float("inf")
    return sum(
        (n_per_expert[k] / total_n) * mase_per_expert[k]
        for k in mase_per_expert
    )


print("Metric functions defined: calc_rmse, calc_mae, calc_mape, calc_mase, calc_all_metrics, weighted_mase_global")

---

## 4. Candidate model families (5 models per family)

### 4.1 Deep-learning wrapper

The deep-learning candidates share one PyTorch module (`_DLModule`) wrapped in a scikit-learn-compatible estimator (`TorchRNNRegressor`), so that every candidate can be evaluated by the same cross-validation engine. Implementation details that matter for interpreting the results:

- Inputs and target are standardised with the mean and standard deviation of the training data of each fit.
- Each observation is presented to the network as a sequence of length 1 with all features as channels, shape `(batch, 1, n_features)`. The LSTM, GRU and Transformer variants therefore act on a single time step (no explicit temporal sequence); temporal information enters only through the lag features. The "TCN" variant applies one 1-D convolution (kernel size 3) along the feature axis followed by global average pooling; the "ANN" variant is a fully connected network.
- Training: Adam optimiser, MSE loss, batch size 256, up to 50 epochs, no shuffling. The last 15 % of the training data (chronologically) is held out for early stopping (patience 5), and the weights with the lowest validation loss are restored.

In [ ]:
class _DLModule(nn.Module):
    """Multi-architecture PyTorch module for tabular forecasting (LSTM, GRU, ANN, TCN, Transformer)."""
    def __init__(self, input_size, hidden_size, num_layers, arch_type="lstm", dropout=0.1):
        super().__init__()
        self.arch_type = arch_type

        if arch_type in ["lstm", "gru"]:
            RNNClass = nn.LSTM if arch_type == "lstm" else nn.GRU
            self.net = RNNClass(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.fc = nn.Linear(hidden_size, 1)

        elif arch_type == "ann":
            layers = []
            curr_size = input_size
            for _ in range(num_layers):
                layers.extend([nn.Linear(curr_size, hidden_size), nn.ReLU(), nn.Dropout(dropout)])
                curr_size = hidden_size
            layers.append(nn.Linear(hidden_size, 1))
            self.net = nn.Sequential(*layers)

        elif arch_type == "tcn":
            self.net = nn.Sequential(
                nn.Conv1d(1, hidden_size, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1)
            )
            self.fc = nn.Linear(hidden_size, 1)

        elif arch_type == "transformer":
            encoder_layer = nn.TransformerEncoderLayer(d_model=input_size, nhead=1, dim_feedforward=hidden_size, batch_first=True)
            self.net = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.fc = nn.Linear(input_size, 1)

    def forward(self, x):
        if self.arch_type in ["lstm", "gru"]:
            out, _ = self.net(x)
            return self.fc(out[:, -1, :]).squeeze(-1)
        elif self.arch_type == "ann":
            return self.net(x.squeeze(1)).squeeze(-1)
        elif self.arch_type == "tcn":
            out = self.net(x)
            return self.fc(out.squeeze(-1)).squeeze(-1)
        elif self.arch_type == "transformer":
            out = self.net(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

class TorchRNNRegressor(BaseEstimator, RegressorMixin):
    """scikit-learn-compatible regression wrapper around the PyTorch module above."""
    def __init__(self, hidden_size=64, num_layers=2, arch_type="lstm", lr=1e-3, epochs=50, batch_size=256, patience=5, dropout=0.1, random_state=42):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.arch_type = arch_type
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.dropout = dropout
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        X, y = np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)
        self.x_mean_, self.x_std_ = X.mean(axis=0), X.std(axis=0) + 1e-8
        self.y_mean_, self.y_std_ = y.mean(), y.std() + 1e-8
        X_t = torch.tensor((X - self.x_mean_) / self.x_std_).unsqueeze(1).to(DEVICE)
        y_t = torch.tensor((y - self.y_mean_) / self.y_std_).to(DEVICE)
        n_val = max(1, int(0.15 * len(X_t)))
        train_ds = TensorDataset(X_t[:-n_val], y_t[:-n_val])
        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=False)
        self.model_ = _DLModule(X.shape[1], self.hidden_size, self.num_layers, self.arch_type, self.dropout).to(DEVICE)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        criterion = nn.MSELoss()
        best_val_loss, patience_counter, best_state = float("inf"), 0, None
        for epoch in range(self.epochs):
            self.model_.train()
            for xb, yb in train_loader:
                optimizer.zero_grad(); pred = self.model_(xb); loss = criterion(pred, yb); loss.backward(); optimizer.step()
            self.model_.eval()
            with torch.no_grad():
                val_loss = criterion(self.model_(X_t[-n_val:]), y_t[-n_val:]).item()
            if val_loss < best_val_loss:
                best_val_loss, patience_counter, best_state = val_loss, 0, deepcopy(self.model_.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= self.patience: break
        if best_state: self.model_.load_state_dict(best_state)
        return self

    def predict(self, X):
        X_t = torch.tensor((np.asarray(X, dtype=np.float32) - self.x_mean_) / self.x_std_).unsqueeze(1).to(DEVICE)
        self.model_.eval()
        with torch.no_grad(): return self.model_(X_t).cpu().numpy() * self.y_std_ + self.y_mean_

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return float(1 - ss_res / ss_tot) if ss_tot != 0 else 0.0

### 4.2 Model catalogue

| Family | Candidates (fixed hyperparameters, no tuning at this stage) |
|:---|:---|
| Linear (`linear`) | Ridge, Lasso, ElasticNet, Bayesian Ridge, SGD regressor (elastic-net penalty); each preceded by a `StandardScaler` |
| Machine learning (`ml`) | LightGBM, XGBoost, Random Forest, Extra Trees, Gradient Boosting |
| Deep learning (`dl`) | LSTM, GRU, ANN (3 dense layers), TCN-style convolution, Transformer encoder |

Hyperparameters are fixed defaults here; they are tuned for the selected models in Notebook 07 (HPO). Each call to the factory functions returns new, unfitted instances.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Model catalogue — 5 models per family
# ══════════════════════════════════════════════════════════════════════════════

def get_linear_models():
    """Return 5 linear models, each wrapped in a Pipeline with a StandardScaler."""
    return {
        "ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
        ]),
        "lasso": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_SEED)),
        ]),
        "elasticnet": Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_SEED)),
        ]),
        "bayesian_ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "sgd": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDRegressor(penalty="elasticnet", max_iter=2000, random_state=RANDOM_SEED)),
        ]),
    }


def get_ml_models():
    """Return 5 machine-learning models (tree-based ensembles)."""
    return {
        "lgbm": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31,
            verbose=-1, random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "xgboost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            verbosity=0, random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "random_forest": RandomForestRegressor(
            n_estimators=200, max_depth=None,
            random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "extra_trees": ExtraTreesRegressor(
            n_estimators=200, max_depth=None,
            random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            random_state=RANDOM_SEED,
        ),
    }

def get_dl_models():
    """Return 5 deep-learning models: LSTM, GRU, ANN (MLP), TCN and Transformer."""
    return {
        "lstm_standard": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="lstm",
            random_state=RANDOM_SEED),
        "gru_standard": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="gru",
            random_state=RANDOM_SEED),
        "ann_dense": TorchRNNRegressor(
            hidden_size=128, num_layers=3, arch_type="ann",
            random_state=RANDOM_SEED),
        "tcn_conv": TorchRNNRegressor(
            hidden_size=64, arch_type="tcn",
            random_state=RANDOM_SEED),
        "transformer_lite": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="transformer",
            random_state=RANDOM_SEED),
    }

# Summary
print("Model families defined:")
print(f"  linear : {list(get_linear_models().keys())}")
print(f"  ml     : {list(get_ml_models().keys())}")
print(f"  dl     : {list(get_dl_models().keys())}")

---

## 5. Walk-forward evaluation engine

Every candidate model is evaluated, separately for each expert, with walk-forward cross-validation using `TimeSeriesSplit` (expanding training window, `n_splits=3`, `gap=0`): in each fold the model is trained on all earlier observations of the expert and tested on the following block, so no future information is used for training. Metrics are averaged across folds (non-finite fold values are ignored). If the expert has too few observations for 3 folds, the number of folds is reduced down to a minimum of 2; if not even 2 folds are possible, all metrics are set to infinity and the candidate cannot be selected.

In [ ]:
# ───────────────────────────────────────────────────────────
# Walk-forward cross-validation engine
# ─────────────────────────────────────────────────────────────

def walk_forward_evaluate(model, X, y, n_splits=3, gap=0):
    """Evaluate a model with walk-forward (expanding-window) cross-validation.

    Parameters
    ----------
    model : scikit-learn-compatible estimator
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    n_splits : int — number of cross-validation folds
    gap : int — number of samples left out between training and test sets to avoid leakage

    Returns
    -------
    dict with keys rmse, mae, mape, mase (averaged across folds)
         plus the list fold_metrics.
    """
    n_samples = len(X)
    effective_n_splits = n_splits

    # Reduce n_splits if needed so that a valid TimeSeriesSplit can be built
    # (TimeSeriesSplit requires n_splits >= 2).
    tscv = None  # initialise tscv outside the loop
    while effective_n_splits >= 2:
        try:
            tscv_candidate = TimeSeriesSplit(n_splits=effective_n_splits, gap=gap)
            first_split = next(iter(tscv_candidate.split(X)), None)
            if first_split is not None:  # a first split could be generated
                tscv = tscv_candidate
                break  # valid TimeSeriesSplit configuration found
            else:
                # no split could be generated with this configuration
                effective_n_splits -= 1
        except ValueError as e:
            # TimeSeriesSplit raises ValueError for infeasible parameters
            effective_n_splits -= 1
        except Exception as e:
            print(f"    [WARNING] Unexpected error while creating/checking TimeSeriesSplit with n_splits={effective_n_splits}: {e}")
            effective_n_splits -= 1

    if tscv is None:  # no valid tscv was found
        print(f"    [WARNING] Too few samples for TimeSeriesSplit (n_samples={n_samples}, gap={gap}; fewer than 2 valid folds). Cross-validation skipped.")
        return {
            "rmse": float("inf"), "mae": float("inf"),
            "mape": float("inf"), "mase": float("inf"),
            "fold_metrics": [],
        }

    fold_metrics = []

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Basic check that training and test sets are not empty
        if len(X_train) == 0 or len(X_test) == 0:
            print(f"    [WARNING] Fold {fold_idx} produced an empty training or test set. Fold skipped.")
            metrics = {
                "fold": fold_idx,
                "rmse": float("inf"),
                "mae": float("inf"),
                "mape": float("inf"),
                "mase": float("inf"),
            }
        else:
            try:
                m = deepcopy(model)
                m.fit(X_train, y_train)
                y_pred = m.predict(X_test)
                metrics = calc_all_metrics(y_test, y_pred)
                metrics["fold"] = fold_idx
            except Exception as e:
                print(f"    [WARNING] Fold {fold_idx} failed during model fit/predict: {e}")
                metrics = {
                    "fold": fold_idx,
                    "rmse": float("inf"),
                    "mae": float("inf"),
                    "mape": float("inf"),
                    "mase": float("inf"),
                }
        fold_metrics.append(metrics)

    # Average across folds
    avg = {}
    for metric in ["rmse", "mae", "mape", "mase"]:
        vals = [fm[metric] for fm in fold_metrics if np.isfinite(fm[metric])]
        avg[metric] = float(np.mean(vals)) if vals else float("inf")

    avg["fold_metrics"] = fold_metrics
    return avg


print("Walk-forward evaluation engine ready (TimeSeriesSplit, n_splits=3, gap=0).")

---

## 6. Bias–variance constraints per expert

Before running the evaluations, the complexity of the candidate models is restricted according to the number of observations $n$ of each expert (cluster combination). Small clusters cannot support high-capacity models without overfitting, so more flexible families are only admitted as the sample size grows:

| Expert size | Allowed candidates | Rationale |
|:---|:---|:---|
| $n < 50$ | Ridge and Lasso only | Very few observations: strong regularisation is needed; tree ensembles have too many degrees of freedom. |
| $50 \le n < 200$ | Linear + ML | Moderate data: regularised linear models are preferred; tree ensembles may still work. DL models would overfit heavily. |
| $200 \le n < 1000$ | Linear + ML | Enough data for ML ensembles but, under the rule of thumb adopted here, not for neural networks. |
| $n \ge 1000$ | All families (linear + ML + DL) | Enough data for deep-learning architectures to be trained. |

The thresholds are design choices (heuristics) of this study. In the code, $n$ is the number of hourly observations of the expert after the merge of Section 2.3.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Bias–variance constraints: allowed model families per expert
# ══════════════════════════════════════════════════════════════════════════════

expert_ids = sorted(df["expert_id"].unique())
n_per_expert = df.groupby("expert_id").size().to_dict()

# Assign the allowed model families according to cluster size
expert_allowed_families = {}
expert_constraints = []

for eid in expert_ids:
    n_obs = n_per_expert[eid]
    if n_obs < 50:
        allowed = ["linear_restricted"]
        constraint = "n < 50: Ridge/Lasso only (maximum regularisation)"
    elif n_obs < 200:
        allowed = ["linear", "ml"]
        constraint = "50 <= n < 200: linear + ML (no DL)"
    elif n_obs < 1000:
        allowed = ["linear", "ml"]
        constraint = "200 <= n < 1000: linear + ML (no DL)"
    else:
        allowed = ["linear", "ml", "dl"]
        constraint = "n >= 1000: all families"

    expert_allowed_families[eid] = allowed
    expert_constraints.append({
        "expert_id": eid,
        "n_obs": n_obs,
        "allowed_families": ", ".join(allowed),
        "constraint": constraint,
    })

df_constraints = pd.DataFrame(expert_constraints)
print("Bias–variance constraints per expert\n")
display(df_constraints)

In [ ]:
def get_models_for_expert(expert_id, family=None):
    """Return the candidate models allowed for an expert, optionally restricted to one family.

    Applies the bias–variance constraints:
      - n < 50:   Ridge and Lasso only
      - n < 1000: no DL
    """
    allowed = expert_allowed_families[expert_id]

    models = {}

    if family is None or family == "linear":
        if "linear_restricted" in allowed:
            # Only Ridge and Lasso for very small clusters
            all_linear = get_linear_models()
            models.update({
                k: v for k, v in all_linear.items()
                if k in ["ridge", "lasso"]
            })
        elif "linear" in allowed:
            if family == "linear" or family is None:
                models.update(get_linear_models())

    if family is None or family == "ml":
        if "ml" in allowed:
            models.update(get_ml_models())

    if family is None or family == "dl":
        if "dl" in allowed:
            models.update(get_dl_models())

    return models


# Show the candidates allowed for each expert
for eid in expert_ids:
    m = get_models_for_expert(eid)
    print(f"Expert {eid} (n={n_per_expert[eid]}): {list(m.keys())}")

---

## 7. Homogeneous evaluation (one model family for all experts)

In the homogeneous strategy the whole MoE is built from a single model family. For each family (linear, ML, DL) and each expert, every candidate of that family that the expert is allowed to use is evaluated with the walk-forward engine on the expert's own observations (rows with missing values are removed first). Within the family, the candidate with the lowest cross-validated RMSE is retained for that expert. Experts for which a family is not allowed by the bias–variance constraints are skipped for that family, so the ML and DL families may cover only a subset of the experts.

The resulting per-expert MASE values are then aggregated into the size-weighted global MASE of each family (Section 7.1).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Homogeneous evaluation: evaluate each family separately for every expert
# ══════════════════════════════════════════════════════════════════════════════

homogeneous_results = []  # list of dicts
homogeneous_best_per_family = {}  # family -> {expert_id: best_model_name}

for family_name in ["linear", "ml", "dl"]:
    print(f"\n{'='*70}")
    print(f"HOMOGENEOUS EVALUATION — Family: {family_name.upper()}")
    print(f"{'='*70}")

    family_best = {}

    for eid in expert_ids:
        n_obs = n_per_expert[eid]
        mask = df["expert_id"] == eid
        X_expert = df.loc[mask, FEATURE_COLS].values
        y_expert = df.loc[mask, TARGET].values

        # Drop rows with NaN
        valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
        X_expert, y_expert = X_expert[valid], y_expert[valid]

        print(f"\n  Expert {eid} (n={len(X_expert)})")

        # Candidate models of this family, respecting the constraints
        if family_name == "linear":
            if "linear_restricted" in expert_allowed_families[eid]:
                all_linear = get_linear_models()
                models = {k: v for k, v in all_linear.items() if k in ["ridge", "lasso"]}
            else:
                models = get_linear_models()
        elif family_name == "ml":
            if "ml" not in expert_allowed_families[eid]:
                print(f"    SKIPPED — ML not allowed for n={n_obs}")
                continue
            models = get_ml_models()
        elif family_name == "dl":
            if "dl" not in expert_allowed_families[eid]:
                print(f"    SKIPPED — DL not allowed for n={n_obs} (requires >= 1000)")
                continue
            models = get_dl_models()

        # Evaluate every model of the family
        best_rmse = float("inf")
        best_model_name = None

        for model_name, model in models.items():
            result = walk_forward_evaluate(model, X_expert, y_expert)
            print(f"    {model_name:25s}  RMSE={result['rmse']:.2f}  MAE={result['mae']:.2f}  MAPE={result['mape']:.4f}")

            homogeneous_results.append({
                "evaluation": "homogeneous",
                "family": family_name,
                "expert_id": eid,
                "n_obs": len(X_expert),
                "model": model_name,
                "rmse": result["rmse"],
                "mae": result["mae"],
                "mape": result["mape"],
                "mase": result["mase"],
            })

            if result["rmse"] < best_rmse:
                best_rmse = result["rmse"]
                best_model_name = model_name

        family_best[eid] = best_model_name
        print(f"    >>> BEST: {best_model_name} (RMSE={best_rmse:.2f})")

    homogeneous_best_per_family[family_name] = family_best

print("\n" + "="*70)
print("Homogeneous evaluation completed.")
print(f"Total results collected: {len(homogeneous_results)}")

### 7.1 Global MASE per homogeneous family

For each family, the best candidate per expert (lowest RMSE) is taken, and its MASE values are combined into $\text{MASE}_{\text{global}}$, weighted by the number of observations of each expert covered by the family. Unweighted averages of RMSE, MAPE (fraction) and MAE across the covered experts are also reported.

In [ ]:
# ── Global MASE of each homogeneous family ─────────────────────────────────────

df_homo = pd.DataFrame(homogeneous_results)

# Guard against an empty df_homo
if df_homo.empty:
    print("WARNING: df_homo is empty. There are no homogeneous evaluation results to process.")
    # Initialise these variables to avoid later errors when there is no data
    homo_mase_global = {}
    homo_summary_rows = []
    # Initialise df_homo_summary as an empty DataFrame to avoid a NameError later
    df_homo_summary = pd.DataFrame(homo_summary_rows)
else:
    homo_mase_global = {}
    homo_summary_rows = []

    for family_name in ["linear", "ml", "dl"]:
        dfam = df_homo[(df_homo["family"] == family_name) & (df_homo["mase"] != np.inf)]
        if dfam.empty:
            continue

        # Best model per expert within the family
        best_idx = dfam.groupby("expert_id")["rmse"].idxmin()
        dfam_best = dfam.loc[best_idx]

        mase_per_expert = dict(zip(dfam_best["expert_id"], dfam_best["mase"]))
        n_expert = dict(zip(dfam_best["expert_id"], dfam_best["n_obs"]))

        mase_g = weighted_mase_global(mase_per_expert, n_expert)
        homo_mase_global[family_name] = mase_g

        avg_rmse = dfam_best["rmse"].mean()
        avg_mape = dfam_best["mape"].mean()
        avg_mae = dfam_best["mae"].mean()

        homo_summary_rows.append({
            "family": family_name,
            "n_experts_covered": len(dfam_best),
            "avg_rmse": avg_rmse,
            "avg_mape": avg_mape,
            "avg_mae": avg_mae,
            "mase_global": mase_g,
        })

        print(f"\nFamily {family_name.upper()}:")
        print(f"  Experts covered: {len(dfam_best)}")
        print(f"  Mean RMSE:       {avg_rmse:.2f}")
        print(f"  Mean MAPE:       {avg_mape:.4f}")
        print(f"  Mean MAE:        {avg_mae:.4f}")

    df_homo_summary = pd.DataFrame(homo_summary_rows)
print("\n" + "="*50)
print("Summary of homogeneous families")
display(df_homo_summary.style.format({"avg_rmse": "{:.2f}", "avg_mape": "{:.6f}", "avg_mae": "{:.6f}"}))

---

## 8. Heterogeneous evaluation (best model per expert across all families)

In the heterogeneous strategy each expert may use a model from any family it is allowed to use (Section 6). For every expert, all allowed candidates (up to 15) are scored with the walk-forward engine; results already obtained in the homogeneous evaluation for the same (expert, model) pair are reused instead of being recomputed, so both strategies are compared on identical cross-validation scores. The candidate with the lowest cross-validated RMSE is selected as the expert's model. This selection is the one persisted in `05_expert_selection_best.csv` and used to train the final experts (Section 10).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Heterogeneous evaluation — evaluate ALL allowed models for each expert
# and select the single best one by RMSE
# ══════════════════════════════════════════════════════════════════════════════

heterogeneous_results = []  # list of dicts
hetero_best_models = {}  # expert_id -> (model_name, family, fitted_model)

print("HETEROGENEOUS EVALUATION — Best model per expert across all families")
print("=" * 70)

for eid in expert_ids:
    n_obs = n_per_expert[eid]
    mask = df["expert_id"] == eid
    X_expert = df.loc[mask, FEATURE_COLS].values
    y_expert = df.loc[mask, TARGET].values

    # Drop rows with NaN
    valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
    X_expert, y_expert = X_expert[valid], y_expert[valid]

    print(f"\n  Expert {eid} (n={len(X_expert)})")

    # ALL allowed models
    models = get_models_for_expert(eid)

    best_rmse = float("inf")
    best_name = None
    best_family = None

    for model_name, model in models.items():
        # Determine the family
        if model_name in get_linear_models():
            fam = "linear"
        elif model_name in get_ml_models():
            fam = "ml"
        else:
            fam = "dl"

        # Reuse the result of the homogeneous evaluation if available
        existing = [
            r for r in homogeneous_results
            if r["expert_id"] == eid and r["model"] == model_name
        ]
        if existing:
            result = {
                "rmse": existing[0]["rmse"],
                "mae": existing[0]["mae"],
                "mape": existing[0]["mape"],
                "mase": existing[0]["mase"],
            }
        else:
            result = walk_forward_evaluate(model, X_expert, y_expert)

        print(f"    {model_name:25s} [{fam:6s}]  RMSE={result['rmse']:.2f}")

        heterogeneous_results.append({
            "evaluation": "heterogeneous",
            "family": fam,
            "expert_id": eid,
            "n_obs": len(X_expert),
            "model": model_name,
            "rmse": result["rmse"],
            "mae": result["mae"],
            "mape": result["mape"],
            "mase": result["mase"],
        })

        if result["rmse"] < best_rmse:
            best_rmse = result["rmse"]
            best_name = model_name
            best_family = fam

    hetero_best_models[eid] = (best_name, best_family)
    print(f"    >>> BEST: {best_name} [{best_family}] (RMSE={best_rmse:.2f})")

print("\n" + "=" * 70)
print("Heterogeneous evaluation completed.")

In [ ]:
# ── Table of the heterogeneous best models ───────────────────────────────────────────

df_hetero = pd.DataFrame(heterogeneous_results)

# Best per expert
best_idx = df_hetero.groupby("expert_id")["rmse"].idxmin()
df_hetero_best = df_hetero.loc[best_idx].sort_values("expert_id").reset_index(drop=True)

print("Heterogeneous selection — best model per expert")
print("=" * 60)
display(
    df_hetero_best[["expert_id", "model", "family", "rmse", "mape", "mase", "n_obs"]]
    .style.format({"rmse": "{:.2f}", "mape": "{:.6f}", "mase": "{:.6f}"})
)

### 8.1 Global MASE: heterogeneous vs. homogeneous strategies

The global MASE of the heterogeneous strategy is computed from the heterogeneous evaluation results and compared with the global MASE of each homogeneous family (Section 7.1). The strategy with the lowest value is highlighted.

In [ ]:
# Global MASE of the heterogeneous evaluation
df_hetero_best_aux = df_hetero[df_hetero["mase"] != np.inf]
hetero_mase_per_expert = dict(zip(df_hetero_best_aux["expert_id"], df_hetero_best_aux["mase"]))
hetero_n_per_expert = dict(zip(df_hetero_best_aux["expert_id"], df_hetero_best_aux["n_obs"]))
hetero_mase_global = weighted_mase_global(hetero_mase_per_expert, hetero_n_per_expert)

print(f"\nHeterogeneous MASE_global = {hetero_mase_global:.4f}")

In [ ]:
# ── Compare heterogeneous vs homogeneous MASE_global ─────────────────────────

comparison_rows = []
for family_name, mase_g in homo_mase_global.items():
    comparison_rows.append({
        "strategy": f"Homogeneous ({family_name})",
        "mase_global": mase_g,
    })
comparison_rows.append({
    "strategy": "Heterogeneous (best per expert)",
    "mase_global": hetero_mase_global,
})

df_comparison = pd.DataFrame(comparison_rows).sort_values("mase_global")

print("\nMASE_global comparison: homogeneous vs heterogeneous")
print("=" * 55)
display(df_comparison.style.format({"mase_global": "{:.6f}"}).highlight_min(subset=["mase_global"]))

---

## 9. Figures

All figures are saved in `RESULTS_DIR` with the prefix `05_`.

### 9.1 Global MASE by selection strategy

Size-weighted global MASE of each homogeneous family and of the heterogeneous strategy (lower is better). File: `05_mase_global_comparison.png`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.1: Global MASE comparison — horizontal bar chart
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

strategies = df_comparison["strategy"].values
mase_vals = df_comparison["mase_global"].values
colors = [PALETTE[i] for i in range(len(strategies))]

bars = ax.barh(strategies, mase_vals, color=colors, edgecolor="black", linewidth=0.5)

# Annotate bars
for bar, val in zip(bars, mase_vals):
    ax.text(
        bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", fontsize=10, fontweight="bold",
    )

ax.set_xlabel("MASE global (weighted by cluster size)")
ax.set_title("Global MASE: Homogeneous vs. Heterogeneous Expert Selection")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mase_global_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

### 9.2 Error per expert: homogeneous families vs. heterogeneous selection

For each expert, the bars show the error of the best candidate of each family (homogeneous evaluation) and the red star marks the error of the model selected by the heterogeneous strategy. A bar of height zero means that the family was not evaluated for that expert (not allowed by the bias–variance constraints). In the MAPE figure the best candidate per family is chosen by MAPE, and MAPE is shown in percent. Files: `05_rmse_per_expert.png`, `05_mape_per_expert.png`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.2a: RMSE per expert — grouped bar chart of the homogeneous families
# ══════════════════════════════════════════════════════════════════════════════

# Get best RMSE per expert per family from homogeneous results
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["rmse"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    rmse_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        rmse_vals.append(row["rmse"].values[0] if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, rmse_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Add heterogeneous best as markers
hetero_rmse = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_rmse.append(row["rmse"].values[0] if not row.empty else 0)
ax.scatter(x, hetero_rmse, marker="*", s=120, color="red", zorder=5, label="Heterogeneous best")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("Expert ID (type_day x type_week)")
ax.set_ylabel("RMSE")
ax.set_title("RMSE per Expert: Homogeneous Families vs. Heterogeneous Best")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.2b: MAPE per expert — grouped bar chart of the homogeneous families
# ══════════════════════════════════════════════════════════════════════════════

# Get best MAPE per expert per family from homogeneous results
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["mape"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    mape_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        mape_vals.append(row["mape"].values[0] * 100 if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, mape_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Add heterogeneous best as markers
hetero_mape = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_mape.append(row["mape"].values[0] * 100 if not row.empty else 0)
ax.scatter(x, hetero_mape, marker="*", s=120, color="red", zorder=5, label="Heterogeneous best")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("Expert ID (type_day x type_week)")
ax.set_ylabel("MAPE %")
ax.set_title("MAPE per Expert: Homogeneous Families vs. Heterogeneous Best")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

### 9.3 Error heatmaps: every candidate model for every expert

Cross-validated RMSE and MAPE (%) of every candidate (rows) for every expert (columns). Empty cells correspond to candidates that were not allowed for that expert. Files: `05_rmse_heatmap.png`, `05_mape_heatmap.png`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.3a: Heatmap — RMSE of every model for every expert
# ══════════════════════════════════════════════════════════════════════════════

# Combine all results
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)

# Pivot for heatmap
pivot = df_all.pivot_table(index="model", columns="expert_id", values="rmse", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "RMSE"},
)
ax.set_title("RMSE Heatmap: Models (rows) vs. Experts (columns)")
ax.set_ylabel("Model")
ax.set_xlabel("Expert ID")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.3b: Heatmap — MAPE (%) of every model for every expert
# ══════════════════════════════════════════════════════════════════════════════

# Combine all results
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)
df_all["mape"] = df_all["mape"] * 100

# Pivot for heatmap
pivot = df_all.pivot_table(index="model", columns="expert_id", values="mape", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "MAPE %"},
)
ax.set_title("MAPE Heatmap: Models (rows) vs. Experts (columns)")
ax.set_ylabel("Model")
ax.set_xlabel("Expert ID")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

### 9.4 Expert size vs. selected model family

Number of observations of each expert against the family of the model selected by the heterogeneous strategy; the vertical lines mark the bias–variance thresholds ($n$ = 50, 200, 1000). File: `05_cluster_size_vs_family.png`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.4: Cluster size vs. selected model family
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))

family_marker = {"linear": "o", "ml": "s", "dl": "D"}
family_color = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, eid in enumerate(expert_ids):
    if eid not in hetero_best_models:
        continue
    model_name, fam = hetero_best_models[eid]
    n = n_per_expert[eid]
    ax.scatter(
        n, i, s=120,
        marker=family_marker.get(fam, "o"),
        color=family_color.get(fam, "gray"),
        edgecolor="black", linewidth=0.5, zorder=5,
    )
    ax.annotate(
        f"  {model_name}", (n, i), fontsize=8, va="center",
    )

# Draw constraint thresholds
for threshold, label, ls in [(50, "n=50", "--"), (200, "n=200", "-."), (1000, "n=1000", ":")]:
    ax.axvline(threshold, color="gray", linestyle=ls, alpha=0.5)
    ax.text(threshold + 5, len(expert_ids) - 0.5, label, fontsize=8, color="gray")

ax.set_yticks(range(len(expert_ids)))
ax.set_yticklabels(expert_ids)
ax.set_xlabel("Number of observations")
ax.set_ylabel("Expert ID")
ax.set_title("Heterogeneous Selection: Cluster Size vs. Best Model Family")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker=m, color="w", markerfacecolor=family_color[f],
           markeredgecolor="black", markersize=10, label=f.upper())
    for f, m in family_marker.items()
]
ax.legend(handles=legend_elements, loc="lower right", title="Family")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_cluster_size_vs_family.png"), dpi=300, bbox_inches="tight")
plt.show()

### 9.5 Error distribution per family (homogeneous strategy)

Distribution across experts of the best RMSE and the best MAPE (fraction) achieved within each family. Files: `05_rmse_boxplot_by_family.png`, `05_mape_boxplot_by_family.png`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.5a: Box plot — RMSE distribution per family
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

# Best RMSE per expert per family (from homogeneous)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["rmse"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "rmse": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="rmse", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="rmse", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Model Family")
ax.set_ylabel("RMSE (best per expert)")
ax.set_title("RMSE Distribution by Family (Homogeneous Strategy)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 9.5b: Box plot — MAPE distribution per family
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

# Best MAPE per expert per family (from homogeneous)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["mape"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "mape": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="mape", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="mape", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Model Family")
ax.set_ylabel("MAPE (best per expert)")
ax.set_title("MAPE Distribution by Family (Homogeneous Strategy)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

---

## 10. Final expert training and persistence

### 10.1 Training the selected experts

For each expert, a new instance of the model selected by the heterogeneous strategy (Section 8) is fitted on **all** of that expert's observations (after removing rows with missing values). These fitted models are the experts consumed by Notebook 06; their hyperparameters are tuned later in Notebook 07.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Train the selected models on ALL the data of each expert
# ══════════════════════════════════════════════════════════════════════════════

final_models = {}  # expert_id -> fitted model

print("Training final models on all the data of each expert...")
print("=" * 60)

for eid in expert_ids:
    if eid not in hetero_best_models:
        continue

    model_name, family = hetero_best_models[eid]

    # Fresh, unfitted instance of the model
    all_models = get_models_for_expert(eid)
    if model_name not in all_models:
        print(f"  WARNING: {model_name} not found for expert {eid}; skipped.")
        continue

    model = deepcopy(all_models[model_name])

    # Prepare data
    mask = df["expert_id"] == eid
    X_expert = df.loc[mask, FEATURE_COLS].values
    y_expert = df.loc[mask, TARGET].values

    valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
    X_expert, y_expert = X_expert[valid], y_expert[valid]

    # Fit on all data
    model.fit(X_expert, y_expert)
    final_models[eid] = {
        "model": model,
        "model_name": model_name,
        "family": family,
        "n_obs": len(X_expert),
    }

    print(f"  Expert {eid}: {model_name} [{family}] trained on {len(X_expert)} observations")

print(f"\nTotal final models: {len(final_models)}")

### 10.2 Saving the results

The evaluation tables, the fitted experts (as one dictionary and as one file per expert) and the strategy comparison are written to disk (see the Outputs table at the top of the notebook).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Save results
# ══════════════════════════════════════════════════════════════════════════════

# 1. Full results CSV
all_results_df = pd.concat([
    pd.DataFrame(homogeneous_results),
    pd.DataFrame(heterogeneous_results),
], ignore_index=True)

csv_path = os.path.join(RESULTS_DIR, "05_expert_selection.csv")
all_results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Total rows: {len(all_results_df)}")

# 2. Best model per expert (summary CSV)
summary_path = os.path.join(RESULTS_DIR, "05_expert_selection_best.csv")
df_hetero_best.to_csv(summary_path, index=False)
print(f"Best-per-expert CSV: {summary_path}")

# 3. Pickle with all fitted final models
pickle_path = os.path.join(MODELS_DIR, "expert_models.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump(final_models, f)
print(f"Models pickle saved: {pickle_path}")

# 4. One file per expert
experts_dir = os.path.join(MODELS_DIR, "experts")

for eid, entry in final_models.items():
    # eid format: "day0_week1"
    fname = f"expert_{eid}.pkl"  # -> expert_day0_week1.pkl
    fpath = os.path.join(experts_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(entry["model"], f)

print(f"Per-expert pkl files saved in: {experts_dir}/")
print(f"  Files: {sorted(os.listdir(experts_dir))}")

# 5. Strategy comparison table
comparison_path = os.path.join(RESULTS_DIR, "05_mase_global_comparison.csv")
df_comparison.to_csv(comparison_path, index=False)
print(f"MASE comparison CSV: {comparison_path}")

### 10.3 Summary

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Final summary
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("EXPERT SELECTION — FINAL SUMMARY")
print("=" * 70)

print(f"\nTotal experts: {len(expert_ids)}")
print(f"Models evaluated per expert: up to 15 (5 linear + 5 ML + 5 DL)")
print(f"Walk-forward cross-validation: TimeSeriesSplit(n_splits=3, gap=0)")

print("\n--- Bias–variance constraints ---")
for _, row in df_constraints.iterrows():
    print(f"  Expert {row['expert_id']:10s}  n={row['n_obs']:6d}  -> {row['constraint']}")

print("\n--- MASE_global ranking ---")
for _, row in df_comparison.iterrows():
    marker = " <<<" if row["mase_global"] == df_comparison["mase_global"].min() else ""
    print(f"  {row['strategy']:40s}  MASE_global = {row['mase_global']:.4f}{marker}")

print("\n--- Best model per expert (heterogeneous) ---")
for eid in expert_ids:
    if eid in hetero_best_models:
        name, fam = hetero_best_models[eid]
        row = df_hetero_best[df_hetero_best["expert_id"] == eid].iloc[0]
        print(f"  Expert {eid:10s}  {name:25s} [{fam:6s}]  RMSE={row['rmse']:.2f}  MAPE={row['mape']:.4f}")

print("\n--- Outputs ---")
print(f"  {csv_path}")
print(f"  {summary_path}")
print(f"  {pickle_path}  (full dictionary, kept for backward compatibility)")
print(f"  {experts_dir}/expert_<eid>.pkl  ({len(final_models)} individual files, read by Notebook 06)")
print(f"  {comparison_path}")
print(f"  {os.path.join(RESULTS_DIR, '05_mase_global_comparison.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_per_expert.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_per_expert.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_heatmap.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_heatmap.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_cluster_size_vs_family.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_boxplot_by_family.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_boxplot_by_family.png')}")
print("\nNotebook 05 complete.")